In [1]:
import time
import hmac
import hashlib
import requests
import os
from dotenv import load_dotenv
from urllib.parse import urlencode

load_dotenv()

# 1. Set up your API Key and Secret
BASE_URL = "https://mock-api.roostoo.com"
API_KEY = os.getenv("TESTING_API_KEY")
SECRET_KEY = os.getenv("TESTING_API_SECRET")

In [2]:
# 2. Define the Base URL and Endpoint
# (Update the base URL to match Roostoo's specific environment if needed)
ENDPOINT = '/v3/exchangeInfo'


def _get_timestamp():
    """Return a 13-digit millisecond timestamp as string."""
    return str(int(time.time() * 1000))


def _get_signed_headers(payload: dict = {}):
    """
    Generate signed headers and totalParams for RCL_TopLevelCheck endpoints.
    """
    payload['timestamp'] = _get_timestamp()
    sorted_keys = sorted(payload.keys())
    total_params = "&".join(f"{k}={payload[k]}" for k in sorted_keys)

    signature = hmac.new(
        SECRET_KEY.encode('utf-8'),
        total_params.encode('utf-8'),
        hashlib.sha256
    ).hexdigest()

    headers = {
        'RST-API-KEY': API_KEY,
        'MSG-SIGNATURE': signature
    }

    return headers, payload, total_params

In [3]:
# ------------------------------
# Public Endpoints
# ------------------------------

def check_server_time():
    """Check API server time."""
    url = f"{BASE_URL}/v3/serverTime"
    try:
        res = requests.get(url)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error checking server time: {e}")
        return None


def get_exchange_info():
    """Get exchange trading pairs and info."""
    url = f"{BASE_URL}/v3/exchangeInfo"
    try:
        res = requests.get(url)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting exchange info: {e}")
        return None


def get_ticker(pair=None):
    """Get ticker for one or all pairs."""
    url = f"{BASE_URL}/v3/ticker"
    params = {'timestamp': _get_timestamp()}
    if pair:
        params['pair'] = pair
    try:
        res = requests.get(url, params=params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting ticker: {e}")
        return None


# ------------------------------
# Signed Endpoints
# ------------------------------

def get_balance():
    """Get wallet balances (RCL_TopLevelCheck)."""
    url = f"{BASE_URL}/v3/balance"
    headers, payload, _ = _get_signed_headers({})
    try:
        res = requests.get(url, headers=headers, params=payload)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting balance: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def get_pending_count():
    """Get total pending order count."""
    url = f"{BASE_URL}/v3/pending_count"
    headers, payload, _ = _get_signed_headers({})
    try:
        res = requests.get(url, headers=headers, params=payload)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error getting pending count: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def place_order(pair_or_coin, side, quantity, price=None, order_type=None):
    """
    Place a LIMIT or MARKET order.
    """
    url = f"{BASE_URL}/v3/place_order"
    pair = f"{pair_or_coin}/USD" if "/" not in pair_or_coin else pair_or_coin

    if order_type is None:
        order_type = "LIMIT" if price is not None else "MARKET"

    if order_type == 'LIMIT' and price is None:
        print("Error: LIMIT orders require 'price'.")
        return None

    payload = {
        'pair': pair,
        'side': side.upper(),
        'type': order_type.upper(),
        'quantity': str(quantity)
    }
    if order_type == 'LIMIT':
        payload['price'] = str(price)

    headers, _, total_params = _get_signed_headers(payload)
    headers['Content-Type'] = 'application/x-www-form-urlencoded'

    try:
        res = requests.post(url, headers=headers, data=total_params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error placing order: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def query_order(order_id=None, pair=None, pending_only=None):
    """Query order history or pending orders."""
    url = f"{BASE_URL}/v3/query_order"
    payload = {}
    if order_id:
        payload['order_id'] = str(order_id)
    elif pair:
        payload['pair'] = pair
        if pending_only is not None:
            payload['pending_only'] = 'TRUE' if pending_only else 'FALSE'

    headers, _, total_params = _get_signed_headers(payload)
    headers['Content-Type'] = 'application/x-www-form-urlencoded'

    try:
        res = requests.post(url, headers=headers, data=total_params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying order: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None


def cancel_order(order_id=None, pair=None):
    """Cancel specific or all pending orders."""
    url = f"{BASE_URL}/v3/cancel_order"
    payload = {}
    if order_id:
        payload['order_id'] = str(order_id)
    elif pair:
        payload['pair'] = pair

    headers, _, total_params = _get_signed_headers(payload)
    headers['Content-Type'] = 'application/x-www-form-urlencoded'

    try:
        res = requests.post(url, headers=headers, data=total_params)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"Error canceling order: {e}")
        print(f"Response text: {e.response.text if e.response else 'N/A'}")
        return None

In [14]:
print("\n--- Checking Server Time ---")
print(check_server_time())

print("\n--- Getting Exchange Info ---")
info = get_exchange_info()
print(info)

print("\n--- Getting Market Ticker (BTC/USD) ---")
ticker = get_ticker("BTC/USD")
if ticker:
    print(ticker.get("Data", {}).get("BTC/USD", {}))


--- Checking Server Time ---
{'ServerTime': 1773736755224}

--- Getting Exchange Info ---
{'IsRunning': True, 'InitialWallet': {'USD': 50000}, 'TradePairs': {'OPEN/USD': {'Coin': 'OPEN', 'CoinFullName': 'OPEN', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 4, 'AmountPrecision': 1, 'MiniOrder': 1}, 'TRUMP/USD': {'Coin': 'TRUMP', 'CoinFullName': 'TRUMP', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 3, 'AmountPrecision': 3, 'MiniOrder': 1}, 'TON/USD': {'Coin': 'TON', 'CoinFullName': 'Toncoin', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 3, 'AmountPrecision': 2, 'MiniOrder': 1}, 'S/USD': {'Coin': 'S', 'CoinFullName': 'S', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 5, 'AmountPrecision': 1, 'MiniOrder': 1}, 'SOL/USD': {'Coin': 'SOL', 'CoinFullName': 'SOL', 'Unit': 'USD', 'UnitFullName': 'US Dollar', 'CanTrade': True, 'PricePrecision': 2, 'AmountPrecis

In [14]:
ticker = get_ticker()
print(ticker)

{'Success': True, 'ErrMsg': '', 'ServerTime': 1773758090027, 'Data': {'1000CHEEMS/USD': {'MaxBid': 0.00052, 'MinAsk': 0.000521, 'LastPrice': 0.000521, 'Change': -0.0207, 'CoinTradeValue': 1566768422, 'UnitTradeValue': 833593.817958}, 'AAVE/USD': {'MaxBid': 121.7, 'MinAsk': 121.71, 'LastPrice': 121.71, 'Change': 0.0148, 'CoinTradeValue': 119847.618, 'UnitTradeValue': 14612454.38131}, 'ADA/USD': {'MaxBid': 0.2877, 'MinAsk': 0.2878, 'LastPrice': 0.2877, 'Change': 0.0123, 'CoinTradeValue': 201042507.2, 'UnitTradeValue': 57707571.29467}, 'APT/USD': {'MaxBid': 0.994, 'MinAsk': 0.995, 'LastPrice': 0.995, 'Change': -0.0188, 'CoinTradeValue': 12566244.27, 'UnitTradeValue': 12602704.51031}, 'ARB/USD': {'MaxBid': 0.1079, 'MinAsk': 0.108, 'LastPrice': 0.1079, 'Change': 0.0065, 'CoinTradeValue': 63418094.8, 'UnitTradeValue': 6911285.74755}, 'ASTER/USD': {'MaxBid': 0.759, 'MinAsk': 0.76, 'LastPrice': 0.759, 'Change': 0.0541, 'CoinTradeValue': 59891623.53, 'UnitTradeValue': 45006355.19019}, 'AVAX/USD

In [ ]:
print(ticker.get("Data", {}).get("BNB/USD", {}))
print(place_order("BNB/USD", "BUY", 0.6116845008481413))      

{'MaxBid': 668.17, 'MinAsk': 668.18, 'LastPrice': 668.17, 'Change': -0.0083, 'CoinTradeValue': 192229.273, 'UnitTradeValue': 129899699.89618}
{'Success': False, 'ErrMsg': 'quantity step size error'}


In [4]:
print(get_balance())

{'Success': True, 'ErrMsg': '', 'SpotWallet': {'BNB': {'Free': 5.968, 'Lock': 0}, 'BTC': {'Free': 0.05376, 'Lock': 0}, 'DOGE': {'Free': 120376, 'Lock': 0}, 'NEAR': {'Free': 0, 'Lock': 0}, 'USD': {'Free': 29494.11, 'Lock': 0}}, 'MarginWallet': {}}


In [ ]:
print(ticker.get("Data", {}).get("BNB/USD", {}))

{}


In [9]:
print(place_order("NEAR/USD", "SELL", 2722.9))

{'Success': True, 'ErrMsg': '', 'OrderDetail': {'Pair': 'NEAR/USD', 'OrderID': 2761399, 'Status': 'FILLED', 'Role': 'TAKER', 'ServerTimeUsage': 0.004042368, 'CreateTimestamp': 1773757689576, 'FinishTimestamp': 1773757689580, 'Side': 'SELL', 'Type': 'MARKET', 'StopType': 'GTC', 'Price': 1.445, 'Quantity': 2722.9, 'FilledQuantity': 2722.9, 'FilledAverPrice': 1.445, 'CoinChange': 2722.9, 'UnitChange': 3934.5905000000002, 'CommissionCoin': 'USD', 'CommissionChargeValue': 3.93459, 'CommissionPercent': 0.001, 'OrderWalletType': 'SPOT', 'OrderSource': 'PUBLIC_API'}}


In [7]:
def sell_all_coins():
    """Fetch balance and sell all coins with a 'Free' balance > 0."""
    print("Fetching wallet balance...")
    balance_resp = get_balance()
    if not balance_resp or not balance_resp.get('Success'):
        print("Failed to retrieve balance.")
        return

    # Get exchange info to ensure we use the correct quantity precision for each coin
    print("Fetching exchange info for precisions...")
    info = get_exchange_info()
    precisions = {}
    if info and info.get('IsRunning'):
        trade_pairs = info.get('TradePairs', {})
        for pair_name, details in trade_pairs.items():
            precisions[pair_name] = details.get('AmountPrecision', 8)

    spot_wallet = balance_resp.get('SpotWallet', {})
    sold_any = False
    
    for coin, data in spot_wallet.items():
        # Skip USD as it's the base currency we are selling into
        if coin == 'USD':
            continue
        
        free_qty = data.get('Free', 0)
        if free_qty > 0:
            # Determine the correct precision for this coin
            pair = f"{coin}/USD"
            precision = precisions.get(pair, 8) 
            rounded_qty = round(float(free_qty), precision)
            
            if rounded_qty > 0:
                print(f"Selling {rounded_qty} {coin}...")
                order_res = place_order(coin, "SELL", rounded_qty)
                print(f"Result for {coin}: {order_res}")
                sold_any = True
            else:
                print(f"Quantity for {coin} ({free_qty}) is too small after rounding to {precision} decimal places.")
    
    if not sold_any:
        print("No coins with positive balance found to sell.")

In [8]:
sell_all_coins()

Fetching wallet balance...
Fetching exchange info for precisions...
Selling 5.968 BNB...
Result for BNB: {'Success': True, 'ErrMsg': '', 'OrderDetail': {'Pair': 'BNB/USD', 'OrderID': 2762433, 'Status': 'FILLED', 'Role': 'TAKER', 'ServerTimeUsage': 0.030633846, 'CreateTimestamp': 1773807804212, 'FinishTimestamp': 1773807804243, 'Side': 'SELL', 'Type': 'MARKET', 'StopType': 'GTC', 'Price': 672.33, 'Quantity': 5.968, 'FilledQuantity': 5.968, 'FilledAverPrice': 672.33, 'CoinChange': 5.968, 'UnitChange': 4012.4654400000004, 'CommissionCoin': 'USD', 'CommissionChargeValue': 4.012465, 'CommissionPercent': 0.001, 'OrderWalletType': 'SPOT', 'OrderSource': 'PUBLIC_API'}}
Selling 0.05376 BTC...
Result for BTC: {'Success': True, 'ErrMsg': '', 'OrderDetail': {'Pair': 'BTC/USD', 'OrderID': 2762434, 'Status': 'FILLED', 'Role': 'TAKER', 'ServerTimeUsage': 0.010492015, 'CreateTimestamp': 1773807804354, 'FinishTimestamp': 1773807804365, 'Side': 'SELL', 'Type': 'MARKET', 'StopType': 'GTC', 'Price': 74356

In [5]:
def get_total_portfolio_value():
    """
    Calculate total wallet value in USD by fetching balances and current market prices.
    Returns a tuple: (total_value_usd, held_coins_list)
    held_coins_list is a list of tuples: (coin, quantity)
    """
    balances = get_balance()
    if not balances or not balances.get('Success'):
        print("Error: Could not fetch balances for portfolio valuation.")
        return None, []

    tickers = get_ticker()
    if not tickers or not tickers.get('Success'):
        print("Error: Could not fetch tickers for portfolio valuation.")
        return None, []

    spot_wallet = balances.get('SpotWallet', {})
    ticker_data = tickers.get('Data', {})
    
    total_value = 0.0
    held_coins = []

    for coin, qty_info in spot_wallet.items():
        total_qty = float(qty_info.get('Free', 0)) + float(qty_info.get('Lock', 0))
        if total_qty <= 0:
            continue

        held_coins.append((coin, total_qty))

        if coin == 'USD':
            total_value += total_qty
        else:
            pair_name = f"{coin}/USD"
            # Handle cases where the ticker might be missing or under a different name
            price_info = ticker_data.get(pair_name)
            if price_info:
                last_price = float(price_info.get('LastPrice', 0))
                total_value += total_qty * last_price
            else:
                print(f"Warning: Price for {coin} ({pair_name}) not found in tickers.")

    return total_value, held_coins

In [9]:
get_total_portfolio_value()

(49584.07, [('USD', 49584.07)])

In [ ]:
def fetch_binance_klines(symbol: str, interval: str, days: int) -> pd.DataFrame:
    """
    Fetch historical kline (candlestick) data from Binance public API.
    No API key required.

    Automatically paginates through the 1000-candle-per-request limit
    so you can fetch arbitrarily long histories (e.g. 30 days of 1m data).

    Returns a DataFrame with columns:
        open_time, open, high, low, close, volume, close_time
    """
    end_ms   = int(time.time() * 1000)
    start_ms = int((time.time() - days * 86400) * 1000)
    limit    = 1000

    all_rows = []
    cursor   = start_ms

    while cursor < end_ms:
        params = {
            "symbol":    symbol,
            "interval":  interval,
            "startTime": cursor,
            "endTime":   end_ms,
            "limit":     limit,
        }
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        batch = resp.json()

        if not batch:
            break

        all_rows.extend(batch)
        # advance cursor past the last candle's open_time
        cursor = batch[-1][0] + 1
        time.sleep(0.1)  # rate-limit courtesy

    cols = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_vol", "trades", "taker_buy_base",
        "taker_buy_quote", "ignore",
    ]
    df = pd.DataFrame(all_rows, columns=cols)
    df["open_time"]  = pd.to_datetime(df["open_time"],  unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    return df[["open_time", "open", "high", "low", "close", "volume", "close_time"]]